In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import linregress

In [3]:
nav = pd.read_csv("../data/processed/nav_history_clean.csv")

scheme = pd.read_csv("../data/processed/scheme_performance_clean.csv")

benchmark = pd.read_csv("../data/processed/benchmark_indices_clean.csv")

In [4]:
nav.head()
scheme.head()
benchmark.head()

,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


In [5]:
nav.columns
scheme.columns
benchmark.columns

Index(['date', 'index_name', 'close_value'], dtype='object')

In [6]:
nav.columns

Index(['amfi_code', 'date', 'nav'], dtype='object')

In [7]:
scheme.columns

Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='object')

In [8]:
benchmark.columns

Index(['date', 'index_name', 'close_value'], dtype='object')

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from scipy.stats import linregress

In [10]:
nav = pd.read_csv("../data/processed/nav_history_clean.csv")
scheme = pd.read_csv("../data/processed/scheme_performance_clean.csv")
benchmark = pd.read_csv("../data/processed/benchmark_indices_clean.csv")

In [11]:
nav["date"] = pd.to_datetime(nav["date"])
benchmark["date"] = pd.to_datetime(benchmark["date"])

In [12]:
nav = nav.sort_values(["amfi_code", "date"])

In [13]:
nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()

In [14]:
nav.head(10)

,amfi_code,date,nav,daily_return
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210
5755,100016,2022-01-10,510.7136,-0.008639
5756,100016,2022-01-11,513.5542,0.005562
5757,100016,2022-01-12,512.3195,-0.002404
5758,100016,2022-01-13,510.2445,-0.004050
5759,100016,2022-01-14,514.3636,0.008073


In [15]:
nav["daily_return"].describe()

count    45960.000000
mean         0.000631
std          0.010290
min         -0.058102
25%         -0.005042
50%          0.000340
75%          0.006324
max          0.064713
Name: daily_return, dtype: float64

In [16]:
fig = px.histogram(
    nav,
    x="daily_return",
    nbins=80,
    title="Distribution of Daily Returns"
)

fig.show()

fig.write_html("../reports/charts/daily_return_distribution.html")
fig.write_image("../reports/charts/daily_return_distribution.png")

In [17]:
cagr = nav.groupby("amfi_code").agg(
    start_nav=("nav", "first"),
    end_nav=("nav", "last"),
    start_date=("date", "first"),
    end_date=("date", "last")
).reset_index()

cagr["years"] = (
    pd.to_datetime(cagr["end_date"]) -
    pd.to_datetime(cagr["start_date"])
).dt.days / 365.25

cagr["cagr"] = (
    (cagr["end_nav"] / cagr["start_nav"])
    ** (1 / cagr["years"]) - 1
)

cagr.head()

,amfi_code,start_nav,end_nav,start_date,end_date,years,cagr
0,100016,520.4608,583.6113,2022-01-03,2026-05-29,4.399726,0.026371
1,100025,26.3169,31.8843,2022-01-03,2026-05-29,4.399726,0.044582
2,100033,107.3758,342.0072,2022-01-03,2026-05-29,4.399726,0.301232
3,101206,305.0996,773.2939,2022-01-03,2026-05-29,4.399726,0.235384
4,101207,38.5736,53.9836,2022-01-03,2026-05-29,4.399726,0.079388


In [18]:
cagr["cagr"].describe()

count    40.000000
mean      0.167357
std       0.103090
min       0.011717
25%       0.068618
50%       0.166073
75%       0.244883
max       0.328274
Name: cagr, dtype: float64

In [19]:
performance = nav.groupby("amfi_code").agg(
    annual_return=("daily_return", lambda x: x.mean() * 252),
    annual_volatility=("daily_return", lambda x: x.std() * np.sqrt(252))
).reset_index()

performance.head()

,amfi_code,annual_return,annual_volatility
0,100016,0.035683,0.145481
1,100025,0.042854,0.039052
2,100033,0.272111,0.189367
3,101206,0.214647,0.145682
4,101207,0.106962,0.257973


In [20]:
risk_free_rate = 0.07

performance["sharpe_ratio"] = (
    (performance["annual_return"] - risk_free_rate)
    / performance["annual_volatility"]
)

performance.head()

,amfi_code,annual_return,annual_volatility,sharpe_ratio
0,100016,0.035683,0.145481,-0.235886
1,100025,0.042854,0.039052,-0.695128
2,100033,0.272111,0.189367,1.067295
3,101206,0.214647,0.145682,0.992892
4,101207,0.106962,0.257973,0.143279


In [21]:
performance["sharpe_ratio"].describe()

count    40.000000
mean      0.427437
std       0.711186
min      -1.802857
25%      -0.023098
50%       0.612768
75%       0.977279
max       1.413064
Name: sharpe_ratio, dtype: float64

In [22]:
fig = px.bar(
    performance.sort_values("sharpe_ratio", ascending=False),
    x="amfi_code",
    y="sharpe_ratio",
    title="Sharpe Ratio by Fund"
)

fig.show()

fig.write_html("../reports/charts/sharpe_ratio.html")
fig.write_image("../reports/charts/sharpe_ratio.png")

In [23]:
performance.shape

(40, 4)

In [24]:
performance.dtypes

amfi_code              int64
annual_return        float64
annual_volatility    float64
sharpe_ratio         float64
dtype: object

In [25]:
performance.isnull().sum()

amfi_code            0
annual_return        0
annual_volatility    0
sharpe_ratio         0
dtype: int64

In [26]:
performance = performance.sort_values("sharpe_ratio", ascending=False)

fig = px.bar(
    performance,
    x="amfi_code",
    y="sharpe_ratio",
    title="Sharpe Ratio by Fund"
)

fig.show()

In [27]:
performance.head(10)

,amfi_code,annual_return,annual_volatility,sharpe_ratio
34,148567,0.270566,0.141937,1.413064
30,120843,0.272602,0.158870,1.275272
36,148569,0.283262,0.176740,1.206640
19,119551,0.231033,0.137414,1.171880
25,120505,0.292653,0.192909,1.154182
38,149323,0.265908,0.177462,1.103947
2,100033,0.272111,0.189367,1.067295
9,118632,0.218037,0.141484,1.046319
3,101206,0.214647,0.145682,0.992892
24,120504,0.212448,0.143638,0.991715


In [28]:
performance["sharpe_ratio"].tolist()[:10]

[1.4130642906503612,
 1.2752716699184306,
 1.2066398927709963,
 1.1718802633787146,
 1.154182307435566,
 1.1039467420861067,
 1.067295002972135,
 1.0463193751356537,
 0.9928920281225743,
 0.9917145076882594]

In [29]:
performance = performance.merge(
    scheme[["amfi_code", "scheme_name"]],
    on="amfi_code",
    how="left"
)

performance.head()

,amfi_code,annual_return,annual_volatility,sharpe_ratio,scheme_name
0,148567,0.270566,0.141937,1.413064,Mirae Asset Large Cap Fund - Regular - Growth
1,120843,0.272602,0.158870,1.275272,Kotak Flexicap Fund - Regular - Growth
2,148569,0.283262,0.176740,1.206640,Mirae Asset Tax Saver Fund - Regular - Growth
3,119551,0.231033,0.137414,1.171880,SBI Bluechip Fund - Regular Plan - Growth
4,120505,0.292653,0.192909,1.154182,ICICI Pru Midcap Fund - Regular - Growth


In [30]:
fig = px.bar(
    performance.sort_values("sharpe_ratio", ascending=False),
    x="scheme_name",
    y="sharpe_ratio",
    title="Sharpe Ratio by Fund"
)

fig.update_layout(
    xaxis_tickangle=-45
)

fig.show()

fig.write_html("../reports/charts/sharpe_ratio.html")
fig.write_image("../reports/charts/sharpe_ratio.png")

In [31]:
performance = performance.merge(
    scheme[["amfi_code", "scheme_name"]],
    on="amfi_code",
    how="left"
)

In [32]:
performance.head()

,amfi_code,annual_return,annual_volatility,sharpe_ratio,scheme_name_x,scheme_name_y
0,148567,0.270566,0.141937,1.413064,Mirae Asset Large Cap Fund - Regular - Growth,Mirae Asset Large Cap Fund - Regular - Growth
1,120843,0.272602,0.158870,1.275272,Kotak Flexicap Fund - Regular - Growth,Kotak Flexicap Fund - Regular - Growth
2,148569,0.283262,0.176740,1.206640,Mirae Asset Tax Saver Fund - Regular - Growth,Mirae Asset Tax Saver Fund - Regular - Growth
3,119551,0.231033,0.137414,1.171880,SBI Bluechip Fund - Regular Plan - Growth,SBI Bluechip Fund - Regular Plan - Growth
4,120505,0.292653,0.192909,1.154182,ICICI Pru Midcap Fund - Regular - Growth,ICICI Pru Midcap Fund - Regular - Growth


In [33]:
def downside_std(returns):
    downside = returns[returns < 0]
    return downside.std() * np.sqrt(252)

In [34]:
downside = (
    nav.groupby("amfi_code")["daily_return"]
    .apply(downside_std)
    .reset_index(name="downside_volatility")
)

downside.head()

,amfi_code,downside_volatility
0,100016,0.083513
1,100025,0.023514
2,100033,0.113229
3,101206,0.083157
4,101207,0.151683


In [35]:
performance = performance.merge(
    downside,
    on="amfi_code",
    how="left"
)

In [36]:
performance["sortino_ratio"] = (
    (performance["annual_return"] - risk_free_rate)
    / performance["downside_volatility"]
)

performance.head()

,amfi_code,annual_return,annual_volatility,sharpe_ratio,scheme_name_x,scheme_name_y,downside_volatility,sortino_ratio
0,148567,0.270566,0.141937,1.413064,Mirae Asset Large Cap Fund - Regular - Growth,Mirae Asset Large Cap Fund - Regular - Growth,0.086168,2.327618
1,120843,0.272602,0.158870,1.275272,Kotak Flexicap Fund - Regular - Growth,Kotak Flexicap Fund - Regular - Growth,0.087806,2.307377
2,148569,0.283262,0.176740,1.206640,Mirae Asset Tax Saver Fund - Regular - Growth,Mirae Asset Tax Saver Fund - Regular - Growth,0.101663,2.097732
3,119551,0.231033,0.137414,1.171880,SBI Bluechip Fund - Regular Plan - Growth,SBI Bluechip Fund - Regular Plan - Growth,0.077576,2.075814
4,120505,0.292653,0.192909,1.154182,ICICI Pru Midcap Fund - Regular - Growth,ICICI Pru Midcap Fund - Regular - Growth,0.112180,1.984782


In [37]:
performance["sortino_ratio"].describe()

count    40.000000
mean      0.706775
std       1.304839
min      -3.716027
25%      -0.038341
50%       1.056255
75%       1.679727
max       2.327618
Name: sortino_ratio, dtype: float64

In [38]:
fig = px.bar(
    performance.sort_values("sortino_ratio", ascending=False),
    x="scheme_name",
    y="sortino_ratio",
    title="Sortino Ratio by Fund"
)

fig.update_layout(xaxis_tickangle=-45)

fig.show()

fig.write_html("../reports/charts/sortino_ratio.html")
fig.write_image("../reports/charts/sortino_ratio.png")

ValueError: Value of 'x' is not the name of a column in 'data_frame'. Expected one of ['amfi_code', 'annual_return', 'annual_volatility', 'sharpe_ratio', 'scheme_name_x', 'scheme_name_y', 'downside_volatility', 'sortino_ratio'] but received: scheme_name

In [39]:
performance.columns

Index(['amfi_code', 'annual_return', 'annual_volatility', 'sharpe_ratio',
       'scheme_name_x', 'scheme_name_y', 'downside_volatility',
       'sortino_ratio'],
      dtype='object')

In [40]:
performance[["scheme_name_x", "scheme_name_y"]].head()

,scheme_name_x,scheme_name_y
0,Mirae Asset Large Cap Fund - Regular - Growth,Mirae Asset Large Cap Fund - Regular - Growth
1,Kotak Flexicap Fund - Regular - Growth,Kotak Flexicap Fund - Regular - Growth
2,Mirae Asset Tax Saver Fund - Regular - Growth,Mirae Asset Tax Saver Fund - Regular - Growth
3,SBI Bluechip Fund - Regular Plan - Growth,SBI Bluechip Fund - Regular Plan - Growth
4,ICICI Pru Midcap Fund - Regular - Growth,ICICI Pru Midcap Fund - Regular - Growth


In [44]:
fig = px.bar(
    performance.sort_values("sortino_ratio", ascending=False),
    x="scheme_name",
    y="sortino_ratio",
    title="Sortino Ratio by Fund"
)

fig.update_layout(
    xaxis_tickangle=-45,
    xaxis_title="Fund",
    yaxis_title="Sortino Ratio"
)

fig.show()

fig.write_html("../reports/charts/sortino_ratio.html")
fig.write_image("../reports/charts/sortino_ratio.png")

In [42]:
performance = performance.drop(columns=["scheme_name_y"])
performance = performance.rename(columns={"scheme_name_x": "scheme_name"})

In [43]:
performance.columns

Index(['amfi_code', 'annual_return', 'annual_volatility', 'sharpe_ratio',
       'scheme_name', 'downside_volatility', 'sortino_ratio'],
      dtype='object')

In [45]:
fig = px.bar(
    performance.sort_values("sortino_ratio", ascending=False),
    x="scheme_name",
    y="sortino_ratio",
    title="Sortino Ratio by Fund"
)

fig.update_layout(
    xaxis_tickangle=-45,
    xaxis_title="Fund",
    yaxis_title="Sortino Ratio"
)

fig.show()

fig.write_html("../reports/charts/sortino_ratio.html")
fig.write_image("../reports/charts/sortino_ratio.png")

In [46]:
benchmark = benchmark.sort_values("date")

benchmark["benchmark_return"] = benchmark.groupby("index_name")[
    "close_value"
].pct_change()

benchmark.head()

,date,index_name,close_value,benchmark_return
0,2022-01-03,NIFTY50,17492.79,NaN
5750,2022-01-03,CRISIL_LIQUID,2281.51,NaN
2300,2022-01-03,NIFTY_MIDCAP150,9721.79,NaN
6900,2022-01-03,CRISIL_GILT,1451.06,NaN
1150,2022-01-03,NIFTY100,17778.24,NaN


In [47]:
benchmark["index_name"].unique()

array(['NIFTY50', 'CRISIL_LIQUID', 'NIFTY_MIDCAP150', 'CRISIL_GILT',
       'NIFTY100', 'NIFTY500', 'BSE_SMALLCAP'], dtype=object)

In [48]:
nifty = benchmark[benchmark["index_name"] == "NIFTY50"].copy()

nifty.head()

,date,index_name,close_value,benchmark_return
0,2022-01-03,NIFTY50,17492.79,NaN
1,2022-01-04,NIFTY50,17689.64,0.011253
2,2022-01-05,NIFTY50,17835.05,0.008220
3,2022-01-06,NIFTY50,17878.51,0.002437
4,2022-01-07,NIFTY50,17759.15,-0.006676


In [49]:
alpha_beta = pd.merge(
    nav,
    nifty[["date", "benchmark_return"]],
    on="date",
    how="inner"
)

alpha_beta.head()

,amfi_code,date,nav,daily_return,benchmark_return
0,100016,2022-01-03,520.4608,NaN,NaN
1,100016,2022-01-04,515.0971,-0.010306,0.011253
2,100016,2022-01-05,521.7239,0.012865,0.008220
3,100016,2022-01-06,515.7880,-0.011377,0.002437
4,100016,2022-01-07,515.1639,-0.001210,-0.006676


In [50]:
alpha_beta = alpha_beta.dropna()

alpha_beta.head()

,amfi_code,date,nav,daily_return,benchmark_return
1,100016,2022-01-04,515.0971,-0.010306,0.011253
2,100016,2022-01-05,521.7239,0.012865,0.008220
3,100016,2022-01-06,515.7880,-0.011377,0.002437
4,100016,2022-01-07,515.1639,-0.001210,-0.006676
5,100016,2022-01-10,510.7136,-0.008639,0.020592


In [51]:
results = []

for fund in alpha_beta["amfi_code"].unique():

    temp = alpha_beta[alpha_beta["amfi_code"] == fund]

    slope, intercept, r_value, p_value, std_err = linregress(
        temp["benchmark_return"],
        temp["daily_return"]
    )

    results.append({
        "amfi_code": fund,
        "beta": slope,
        "alpha": intercept
    })

alpha_beta_metrics = pd.DataFrame(results)

alpha_beta_metrics.head()

,amfi_code,beta,alpha
0,100016,-0.025909,0.000144
1,100025,-0.016176,0.000171
2,100033,-0.011200,0.001081
3,101206,0.033814,0.000849
4,101207,-0.059856,0.000429


In [52]:
performance = performance.merge(
    alpha_beta_metrics,
    on="amfi_code"
)

performance.head()

,amfi_code,annual_return,annual_volatility,sharpe_ratio,scheme_name,downside_volatility,sortino_ratio,beta,alpha
0,148567,0.270566,0.141937,1.413064,Mirae Asset Large Cap Fund - Regular - Growth,0.086168,2.327618,-0.028133,0.001076
1,120843,0.272602,0.158870,1.275272,Kotak Flexicap Fund - Regular - Growth,0.087806,2.307377,-0.008737,0.001082
2,148569,0.283262,0.176740,1.206640,Mirae Asset Tax Saver Fund - Regular - Growth,0.101663,2.097732,-0.010201,0.001125
3,119551,0.231033,0.137414,1.171880,SBI Bluechip Fund - Regular Plan - Growth,0.077576,2.075814,-0.056045,0.000921
4,120505,0.292653,0.192909,1.154182,ICICI Pru Midcap Fund - Regular - Growth,0.112180,1.984782,-0.017391,0.001163


In [53]:
performance[["alpha", "beta"]].describe()

,alpha,beta
count,40.000000,40.000000
mean,0.000631,0.001073
std,0.000347,0.035890
min,0.000115,-0.061476
25%,0.000274,-0.016480
50%,0.000644,-0.001505
75%,0.000877,0.013191
max,0.001195,0.132608


In [54]:
def max_drawdown(nav_series):
    rolling_max = nav_series.cummax()
    drawdown = (nav_series - rolling_max) / rolling_max
    return drawdown.min()

In [55]:
drawdown = (
    nav.groupby("amfi_code")["nav"]
    .apply(max_drawdown)
    .reset_index(name="max_drawdown")
)

drawdown.head()

,amfi_code,max_drawdown
0,100016,-0.247344
1,100025,-0.043083
2,100033,-0.162172
3,101206,-0.112916
4,101207,-0.354469


In [56]:
performance = performance.merge(
    drawdown,
    on="amfi_code"
)

performance.head()

,amfi_code,annual_return,annual_volatility,sharpe_ratio,scheme_name,downside_volatility,sortino_ratio,beta,alpha,max_drawdown
0,148567,0.270566,0.141937,1.413064,Mirae Asset Large Cap Fund - Regular - Growth,0.086168,2.327618,-0.028133,0.001076,-0.112657
1,120843,0.272602,0.158870,1.275272,Kotak Flexicap Fund - Regular - Growth,0.087806,2.307377,-0.008737,0.001082,-0.129740
2,148569,0.283262,0.176740,1.206640,Mirae Asset Tax Saver Fund - Regular - Growth,0.101663,2.097732,-0.010201,0.001125,-0.163967
3,119551,0.231033,0.137414,1.171880,SBI Bluechip Fund - Regular Plan - Growth,0.077576,2.075814,-0.056045,0.000921,-0.150124
4,120505,0.292653,0.192909,1.154182,ICICI Pru Midcap Fund - Regular - Growth,0.112180,1.984782,-0.017391,0.001163,-0.181885


In [57]:
performance["max_drawdown"].describe()

count    40.000000
mean     -0.178729
std       0.112686
min      -0.525742
25%      -0.215927
50%      -0.163070
75%      -0.117653
max      -0.000977
Name: max_drawdown, dtype: float64

In [58]:
fig = px.bar(
    performance.sort_values("max_drawdown"),
    x="scheme_name",
    y="max_drawdown",
    title="Maximum Drawdown by Fund"
)

fig.update_layout(
    xaxis_tickangle=-45,
    xaxis_title="Fund",
    yaxis_title="Maximum Drawdown"
)

fig.show()

fig.write_html("../reports/charts/max_drawdown.html")
fig.write_image("../reports/charts/max_drawdown.png")

In [59]:
performance["cagr"] = performance["annual_return"]

performance["rank_cagr"] = performance["cagr"].rank(ascending=False)

performance["rank_sharpe"] = performance["sharpe_ratio"].rank(ascending=False)

performance["rank_sortino"] = performance["sortino_ratio"].rank(ascending=False)

performance["rank_alpha"] = performance["alpha"].rank(ascending=False)

performance["rank_drawdown"] = performance["max_drawdown"].rank(ascending=False)

In [60]:
performance["overall_score"] = (
    performance["rank_cagr"] +
    performance["rank_sharpe"] +
    performance["rank_sortino"] +
    performance["rank_alpha"] +
    performance["rank_drawdown"]
)

performance.head()

,amfi_code,annual_return,annual_volatility,sharpe_ratio,scheme_name,downside_volatility,sortino_ratio,beta,alpha,max_drawdown,cagr,rank_cagr,rank_sharpe,rank_sortino,rank_alpha,rank_drawdown,overall_score
0,148567,0.270566,0.141937,1.413064,Mirae Asset Large Cap Fund - Regular - Growth,0.086168,2.327618,-0.028133,0.001076,-0.112657,0.270566,7.0,1.0,1.0,7.0,8.0,24.0
1,120843,0.272602,0.158870,1.275272,Kotak Flexicap Fund - Regular - Growth,0.087806,2.307377,-0.008737,0.001082,-0.129740,0.272602,5.0,2.0,2.0,5.0,13.0,27.0
2,148569,0.283262,0.176740,1.206640,Mirae Asset Tax Saver Fund - Regular - Growth,0.101663,2.097732,-0.010201,0.001125,-0.163967,0.283262,4.0,3.0,3.0,4.0,21.0,35.0
3,119551,0.231033,0.137414,1.171880,SBI Bluechip Fund - Regular Plan - Growth,0.077576,2.075814,-0.056045,0.000921,-0.150124,0.231033,10.0,4.0,4.0,10.0,17.0,45.0
4,120505,0.292653,0.192909,1.154182,ICICI Pru Midcap Fund - Regular - Growth,0.112180,1.984782,-0.017391,0.001163,-0.181885,0.292653,3.0,5.0,5.0,3.0,25.0,41.0


In [61]:
scorecard = performance.sort_values("overall_score")

scorecard.head(10)

,amfi_code,annual_return,annual_volatility,sharpe_ratio,scheme_name,downside_volatility,sortino_ratio,beta,alpha,max_drawdown,cagr,rank_cagr,rank_sharpe,rank_sortino,rank_alpha,rank_drawdown,overall_score
0,148567,0.270566,0.141937,1.413064,Mirae Asset Large Cap Fund - Regular - Growth,0.086168,2.327618,-0.028133,0.001076,-0.112657,0.270566,7.0,1.0,1.0,7.0,8.0,24.0
1,120843,0.272602,0.158870,1.275272,Kotak Flexicap Fund - Regular - Growth,0.087806,2.307377,-0.008737,0.001082,-0.129740,0.272602,5.0,2.0,2.0,5.0,13.0,27.0
2,148569,0.283262,0.176740,1.206640,Mirae Asset Tax Saver Fund - Regular - Growth,0.101663,2.097732,-0.010201,0.001125,-0.163967,0.283262,4.0,3.0,3.0,4.0,21.0,35.0
4,120505,0.292653,0.192909,1.154182,ICICI Pru Midcap Fund - Regular - Growth,0.112180,1.984782,-0.017391,0.001163,-0.181885,0.292653,3.0,5.0,5.0,3.0,25.0,41.0
3,119551,0.231033,0.137414,1.171880,SBI Bluechip Fund - Regular Plan - Growth,0.077576,2.075814,-0.056045,0.000921,-0.150124,0.231033,10.0,4.0,4.0,10.0,17.0,45.0
6,100033,0.272111,0.189367,1.067295,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,0.113229,1.784976,-0.011200,0.001081,-0.162172,0.272111,6.0,7.0,8.0,6.0,20.0,47.0
5,149323,0.265908,0.177462,1.103947,DSP Midcap Fund - Regular - Growth,0.107145,1.828435,0.003479,0.001055,-0.172481,0.265908,8.0,6.0,6.0,8.0,22.0,50.0
8,101206,0.214647,0.145682,0.992892,ABSL Frontline Equity Fund - Regular - Growth,0.083157,1.739436,0.033814,0.000849,-0.112916,0.214647,12.0,9.0,10.0,12.0,9.0,52.0
9,120504,0.212448,0.143638,0.991715,ICICI Pru Bluechip Fund - Direct - Growth,0.081675,1.744076,0.017025,0.000842,-0.125883,0.212448,13.0,10.0,9.0,13.0,12.0,57.0
7,118632,0.218037,0.141484,1.046319,Nippon India Large Cap Fund - Regular - Growth,0.082717,1.789685,0.036434,0.000862,-0.174141,0.218037,11.0,8.0,7.0,11.0,23.0,60.0


In [62]:
fig = px.bar(
    scorecard.head(10),
    x="scheme_name",
    y="overall_score",
    title="Top 10 Mutual Funds Scorecard"
)

fig.update_layout(
    xaxis_tickangle=-45,
    xaxis_title="Fund",
    yaxis_title="Overall Score (Lower is Better)"
)

fig.show()

fig.write_html("../reports/charts/fund_scorecard.html")
fig.write_image("../reports/charts/fund_scorecard.png")

In [63]:
performance.to_csv(
    "../data/processed/fund_performance_metrics.csv",
    index=False
)

In [64]:
scorecard.to_csv(
    "../data/processed/fund_scorecard.csv",
    index=False
)

In [65]:
top5 = scorecard.head(5)["amfi_code"].tolist()

top5

[148567, 120843, 148569, 120505, 119551]

In [66]:
benchmark_filtered = benchmark[
    benchmark["index_name"].isin(["NIFTY50", "NIFTY100"])
].copy()

benchmark_filtered.head()

,date,index_name,close_value,benchmark_return
0,2022-01-03,NIFTY50,17492.79,NaN
1150,2022-01-03,NIFTY100,17778.24,NaN
1,2022-01-04,NIFTY50,17689.64,0.011253
1151,2022-01-04,NIFTY100,17537.52,-0.013540
1152,2022-01-05,NIFTY100,17607.73,0.004003


In [67]:
top5_nav = nav[nav["amfi_code"].isin(top5)].copy()

top5_nav = top5_nav.merge(
    scheme[["amfi_code", "scheme_name"]],
    on="amfi_code",
    how="left"
)

top5_nav.head()

,amfi_code,date,nav,daily_return,scheme_name
0,119551,2022-01-03,54.3856,NaN,SBI Bluechip Fund - Regular Plan - Growth
1,119551,2022-01-04,54.3474,-0.000702,SBI Bluechip Fund - Regular Plan - Growth
2,119551,2022-01-05,54.6869,0.006247,SBI Bluechip Fund - Regular Plan - Growth
3,119551,2022-01-06,55.4550,0.014045,SBI Bluechip Fund - Regular Plan - Growth
4,119551,2022-01-07,55.3692,-0.001547,SBI Bluechip Fund - Regular Plan - Growth


In [68]:
import plotly.express as px

fig = px.line(
    top5_nav,
    x="date",
    y="nav",
    color="scheme_name",
    title="Top 5 Mutual Funds NAV Comparison"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="NAV"
)

fig.show()

fig.write_html("../reports/charts/top5_funds_comparison.html")
fig.write_image("../reports/charts/top5_funds_comparison.png")

In [69]:
fig = px.line(
    benchmark_filtered,
    x="date",
    y="close_value",
    color="index_name",
    title="NIFTY50 vs NIFTY100"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Index Value"
)

fig.show()

fig.write_html("../reports/charts/benchmark_comparison.html")
fig.write_image("../reports/charts/benchmark_comparison.png")

In [ ]:
git add .
git commit -m "Day 4: Added fund performance analytics and benchmark comparison"
git push origin main